# HMM Part-of-Speech Tagger with Viterbi Decoding

A Hidden Markov Model built and trained from scratch on the Brown corpus for part-of-speech tagging, using the Viterbi algorithm for optimal tag-sequence decoding. Compares HMMs trained with three different smoothing schemes: MLE, ELE (Expected Likelihood Estimation), and Lidstone smoothing with α=0.01.

**Corpus:** Brown corpus (categories: news, editorial, reviews) with the universal POS tagset.

**Techniques:** HMM training, Viterbi decoding, Laplace / add-α smoothing comparison, confusion-matrix and per-tag precision / recall / F1 evaluation.

**Libraries:** `nltk` (Brown corpus, `HiddenMarkovModelTrainer`, `LidstoneProbDist` / `MLEProbDist` / `ELEProbDist`), `sklearn` for train/test split and metrics.

## Preparing the environment

Load the Brown corpus and universal tagset.

In [4]:
# Import needed libraries
import nltk
nltk.download('brown')
nltk.download('universal_tagset')
from nltk.corpus import brown

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\<user>\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\<user>\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


## Loading and exploring the data

Load tagged sentences from the news, editorial, and reviews categories of the Brown corpus.

In [5]:
# Load the Brown corpus with the universal tagset for analysis
browndata_tagged_sents = brown.tagged_sents(categories=['news', 'editorial', 'reviews'], tagset='universal')

## Data preprocessing

Lowercase all tokens while preserving the (word, tag) tuple structure.

In [6]:
# Lowercase the words while preserving (word, tag) structure - we use word, tag as specified in the instructions
lowercased_brown_tagsents = [[(word.lower(), tag) for (word, tag) in sent] for sent in browndata_tagged_sents]

## Train / test split

80/20 stratified split of tagged sentences.

In [7]:
# Import needed libraries
from sklearn.model_selection import train_test_split

# Split data: 80% training, 20% testing. And setting the random state to 42 for reproducibility
train_data, test_data = train_test_split(lowercased_brown_tagsents, test_size = 0.2, random_state = 42)

## Training the Hidden Markov Model

Build the vocabulary and tag inventories, then train HMMs with three different smoothing schemes to compare their effect on the Viterbi decoder's output.

In [8]:
# Reference: https://towardsdatascience.com/hidden-markov-models-explained-with-a-real-life-example-and-python-code-2df2a7956d65/
# Flatten all (word, tag) pairs in training data
flattened_train = [word_tag_pair for sent in train_data for word_tag_pair in sent] # loops through every sentence and every pair inside each sentence

# Build set of unique words
unique_words = list(set([word for word, tag in flattened_train])) # extract the word from each (word, tag) pair and convert to a set to remove duplicates.

# Build set of unique tags
unique_tags = list(set([tag for word, tag in flattened_train])) # extract the tag from each (word, tag) pair for uniqueness,then convert to list

### Define smoothing distributions

MLE, ELE (Expected Likelihood Estimation / Laplace-1), and Lidstone with α=0.01.

In [9]:
from nltk.probability import LidstoneProbDist, MLEProbDist, ELEProbDist

def lidstone_prob_dist_001(fd, bins):
    return LidstoneProbDist(fd, 0.01)

def lidstone_prob_dist_01(fd, bins):
    return LidstoneProbDist(fd, 0.1)

def MLE_ProbDist(fd, bins):
    return MLEProbDist(fd)

def ELE_ProbDist(fd, bins):
    return ELEProbDist(fd)

### Train the three HMM variants

In [12]:
#Reference: https://tedboy.github.io/nlps/generated/generated/nltk.HiddenMarkovModelTrainer.html

# Importing necessary modules from NLTK for HMM training and probability distributions
from nltk.tag import HiddenMarkovModelTrainer
from nltk.probability import LidstoneProbDist, MLEProbDist, ELEProbDist

# Create the trainer using tagset and dictionary we made before (so in other words, the unqiue tags and words essentially)
trainer = HiddenMarkovModelTrainer(states = unique_tags, symbols = unique_words)

# 1. Pure HMM (no smoothing) - use train_supervised method as specificed and train_data
hmm_pure = trainer.train_supervised(train_data)

# 2. HMM with Lidstone smoothing (gamma = 0.01 which was given to us to use) - use train_supervised method as specificed and train_data
hmm_lidstone_001 = trainer.train_supervised(train_data, estimator = lidstone_prob_dist_001)

# 3. HMM with Lidstone smoothing (gamma = 0.1 which was given to us to use) - use train_supervised method as specificed and train_data
hmm_lidstone_01 = trainer.train_supervised(train_data, estimator = lidstone_prob_dist_01)

# 4. HMM with MLE (Maximum Likelihood Estimation which was given to us to use) - use train_supervised method as specificed and train_data
hmm_mle = trainer.train_supervised(train_data, estimator = MLE_ProbDist)

# 5. HMM with ELE (Expected Likelihood Estimation which was given to us to use) - use train_supervised method as specificed and train_data
hmm_ele = trainer.train_supervised(train_data, estimator  =ELE_ProbDist)

## Applying the HMM with the Viterbi Algorithm

Use each trained HMM's `best_path` method (Viterbi decoding) to predict tag sequences on the held-out test set.

In [13]:
# reference: https://www.nltk.org/api/nltk.tag.hmm.html#nltk.tag.hmm.HiddenMarkovModelTagger.best_path
# Extract just the words from the test_data - similar what I did with flatten train
test_sentences = [[word for word, tag in sent] for sent in test_data]

# Predict using each of the models we trained before
# 1. Pure HMM (no smoothing)
pred_tags_pure = [hmm_pure.best_path(sent) for sent in test_sentences] # returns the most probable tag sequence

# 2. HMM with Lidstone smoothing (gamma = 0.01)
pred_tags_lidstone_001 = [hmm_lidstone_001.best_path(sent) for sent in test_sentences] # returns the most probable tag sequence

# 3. HMM with Lidstone smoothing (gamma = 0.1)
pred_tags_lidstone_01 = [hmm_lidstone_01.best_path(sent) for sent in test_sentences] # returns the most probable tag sequence

# 4. HMM with MLE
pred_tags_mle = [hmm_mle.best_path(sent) for sent in test_sentences] # returns the most probable tag sequence

# 5. HMM with ELE
pred_tags_ele = [hmm_ele.best_path(sent) for sent in test_sentences] # returns the most probable tag sequence

C:\Users\<user>\anaconda3\lib\site-packages\nltk\tag\hmm.py:334: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
C:\Users\<user>\anaconda3\lib\site-packages\nltk\tag\hmm.py:336: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
C:\Users\<user>\anaconda3\lib\site-packages\nltk\tag\hmm.py:364: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


## Model evaluation

Compare all three smoothing variants on accuracy, per-tag precision/recall/F1, and confusion matrix.

In [15]:
from nltk.metrics import ConfusionMatrix
import itertools

def printConfusionMatrix(labels_predicted, labels_correct):
    actual_tags = list(itertools.chain(*[[tag for word, tag in sent] for sent in labels_correct]))
    predicted_tags = list(itertools.chain(*[[tag for word, tag in sent] for sent in labels_predicted]))
    conf_matrix = ConfusionMatrix(actual_tags, predicted_tags)
    print(conf_matrix)


In [16]:
# Importing necessary libraries for model evaluation and metrics calculation
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelBinarizer
from itertools import chain
import numpy as np

# Defining the conlleval function for evaluating NLP models
def printConlleval(labels_predicted, labels_correct):
    lb = LabelBinarizer() # Initializing the LabelBinarizer for handling label encoding

    # Flattening the list of labels for correct and predicted
    labels_correct_flattened = [(word, tag) for sent in labels_correct for word, tag in sent]
    labels_predicted_flattened = [(word, tag) for sent in labels_predicted for word, tag in list(sent)]

    # Transforming the labels into a binary format for evaluation
    y_true_combined = lb.fit_transform([tag for _, tag in labels_correct_flattened])
    y_pred_combined = lb.transform([tag for _, tag in labels_predicted_flattened])

    tagset = set(lb.classes_)
    tagset = sorted(tagset, key=lambda tag: tag.split('-', 1)[::-1])
    class_indices = {cls: idx for idx, cls in enumerate(lb.classes_)}

    num_sentences = len(labels_predicted)
    total_tokens = sum(len(s) for s in labels_predicted)

    num_correct_sentences, total_correct_tokens = 0, 0
    for pred, true in zip(labels_predicted, labels_correct):
        if len(pred) == len(true):
            correct_tokens = sum(p == t for p, t in zip(pred, true))
            total_correct_tokens += correct_tokens
            if correct_tokens == len(pred):
                num_correct_sentences += 1

    correct_sentences_percentage = num_correct_sentences / num_sentences * 100
    total_correct_tokens_percentage = total_correct_tokens / total_tokens * 100

    classification_report_dict = classification_report(
        y_true_combined,
        y_pred_combined,
        labels=[class_indices[cls] for cls in tagset],
        target_names=tagset,
        output_dict=True,
        zero_division=1
    )


    classification_report_dict.pop('macro avg', None)
    classification_report_dict.pop('weighted avg', None)
    classification_report_dict.pop('samples avg', None)
    classification_report_dict.pop('micro avg', None)

    total_precision = precision_score(y_true_combined, y_pred_combined, average='weighted', zero_division=1)
    total_recall = recall_score(y_true_combined, y_pred_combined, average='weighted', zero_division=1)
    total_f1 = f1_score(y_true_combined, y_pred_combined, average='weighted', zero_division=1)
    total_line = f"{'Total':<15s} {total_precision:<10.2f} {total_recall:<10.2f} {total_f1:<10.2f}"

    report_lines = [f"{k:<15s} {classification_report_dict[k]['precision']:<10.2f} {classification_report_dict[k]['recall']:<10.2f} {classification_report_dict[k]['f1-score']:<10.2f}" for k in classification_report_dict if isinstance(classification_report_dict[k], dict)]
    report_lines.insert(0, "\n")
    report_lines.insert(1, f"{'TAG':<15s} {'Precision':<10s} {'Recall':<10s} {'F1-score':<10s}\n")
    report_lines.insert(2, total_line)
    report_lines.insert(3, '-'*50 + '\n')
    classification_report_str = "\n".join(report_lines)

    additional_info_str = ''
    additional_info_str += f'Total tokens: {total_tokens}\n'
    additional_info_str += f'Total correct tokens: {total_correct_tokens} ({total_correct_tokens_percentage:.2f}%)\n'
    additional_info_str += f'Processed sentences: {num_sentences}\n'
    additional_info_str += f'Completely correct sentences: {num_correct_sentences} ({correct_sentences_percentage:.2f}%)\n'

    print(additional_info_str + classification_report_str)

In [17]:
# Rebuild the test set as wordtag pairs
labels_correct = test_data  # already in the correct format

# Rebuild predicted labels for each model into wordtag pairs by zipping the original sentence with the predicted tags
# Overview of this process (same for every labels_prediction on each model):
# Constructing a list of predicted (word, tag) pairs for each sentence:
# sent is a list of words from the test set
# tags is the list of tags predicted by the model for a sentence
# then we zip them together to get (word, predicted_tag) pairs

# Pure HMM model
labels_predicted_pure = [[(word, tag) for word, tag in zip(sent, tags)] 
                         for sent, tags in zip(test_sentences, pred_tags_pure)]

# Print the results for Pure HMM Model
print("\n---- PURE HMM MODEL EVALUATION ----")
printConlleval(labels_predicted_pure, labels_correct)
printConfusionMatrix(labels_predicted_pure, labels_correct)


# Lidstone 0.01 model
labels_predicted_lidstone_001 = [[(word, tag) for word, tag in zip(sent, tags)] 
                                 for sent, tags in zip(test_sentences, pred_tags_lidstone_001)]

# Print the results for Lidstone 0.01 model
print("\n---- LIDSTONE 0.01 HMM MODEL EVALUATION ----")
printConlleval(labels_predicted_lidstone_001, labels_correct)
printConfusionMatrix(labels_predicted_lidstone_001, labels_correct)


# Lidstone 0.1 model
labels_predicted_lidstone_01 = [[(word, tag) for word, tag in zip(sent, tags)] 
                                for sent, tags in zip(test_sentences, pred_tags_lidstone_01)]

# Print the results for Lidstone 0.1 model
print("\n---- LIDSTONE 0.1 HMM MODEL EVALUATION ----")
printConlleval(labels_predicted_lidstone_01, labels_correct)
printConfusionMatrix(labels_predicted_lidstone_01, labels_correct)


# MLE model
labels_predicted_mle = [[(word, tag) for word, tag in zip(sent, tags)] 
                        for sent, tags in zip(test_sentences, pred_tags_mle)]

# Print the results for MLE model
print("\n---- MLE HMM MODEL EVALUATION ----")
printConlleval(labels_predicted_mle, labels_correct)
printConfusionMatrix(labels_predicted_mle, labels_correct)


# ELE model
labels_predicted_ele = [[(word, tag) for word, tag in zip(sent, tags)] 
                        for sent, tags in zip(test_sentences, pred_tags_ele)]

# Print the results for ELE model
print("\n---- ELE HMM MODEL EVALUATION ----")
printConlleval(labels_predicted_ele, labels_correct)
printConfusionMatrix(labels_predicted_ele, labels_correct)


---- PURE HMM MODEL EVALUATION ----
Total tokens: 40434
Total correct tokens: 20735 (51.28%)
Processed sentences: 1875
Completely correct sentences: 489 (26.08%)


TAG             Precision  Recall     F1-score  

Total           0.97       0.51       0.67      
--------------------------------------------------

.               1.00       0.45       0.62      
ADJ             0.93       0.47       0.62      
ADP             0.96       0.52       0.67      
ADV             0.91       0.54       0.67      
CONJ            0.99       0.48       0.65      
DET             0.99       0.59       0.74      
NOUN            0.97       0.47       0.64      
NUM             0.98       0.53       0.69      
PRON            0.96       0.64       0.77      
PRT             0.90       0.53       0.67      
VERB            0.98       0.55       0.71      
X               0.00       0.97       0.00      
     |                        C         N         P         V      |
     |         A    A    A 

## Persisting the best model

Serialize the best-performing HMM with `dill` (which handles NLTK's nested objects that `pickle` can't) so the trained tagger can be reused without retraining.

In [18]:
# Importing dill library for model serialization
import dill
mybestmodel = hmm_lidstone_001

# serialization with dill
with open('mybestmodel.dill', 'wb') as file:
    dill.dump(mybestmodel, file)

---

## Summary

This project implements an end-to-end HMM POS tagger from foundational NLP primitives rather than a pre-built library abstraction. The Viterbi algorithm handles the exponentially large tag-sequence search space in polynomial time by exploiting the Markov assumption, and the three-way smoothing comparison shows how the choice of probability estimator affects tag prediction on unseen words.